### Generate Synthetic Data

In [ ]:
import numpy as np
import random
import data_processing.data_loading as dl

def get_data_synthetic(num_samples=1000):
    """
    Generates a synthetic dataset with 1K samples and two dimensions.

    Process:
    - Randomly sample class label y from {0, 1}.
    - Sample x ~ N(mu_y, Sigma_y).
    - mu0 = [-2, -2], mu1 = [2, 2]
    - Sigma0 = Sigma1 = 1 * I
    """
    random.seed(0)
    # Parameters
    mu0 = np.array([-3, -3])
    mu1 = np.array([3, 3])
    cov = 5 * np.eye(2)

    # Generate labels (assuming balanced classes for 1K samples)
    n0 = num_samples // 2
    n1 = num_samples - n0

    # Generate X conditioned on y
    X0 = np.random.multivariate_normal(mu0, cov, n0)
    X1 = np.random.multivariate_normal(mu1, cov, n1)

    # Concatenate
    X = np.vstack((X0, X1))
    y = np.array([0] * n0 + [1] * n1)

    # Shuffle the dataset
    indices = np.random.permutation(num_samples)
    X = X[indices]
    y = y[indices]

    return X, y

def load_data(dataset):
    if(dataset=='synthetic'):
      return get_data_synthetic()
    elif(dataset=='polish'):
      return dl.load_polish_notebook()
    elif(dataset == 'compas'):
      return dl.load_compas_notebook()
    elif(dataset == 'german'):
      return dl.load_germanc()

### Different Model Classes Learnt from Data

In [ ]:
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Define Logistic Regression Model
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim, seed=None):
        super(LogisticRegressionModel, self).__init__()
        if seed is not None:
            torch.manual_seed(seed)
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# Define Neural Network Models
class NeuralNetworkModelSingleLayer(nn.Module):
    def __init__(self, input_dim, hidden_nodes=20, seed=None):
        super(NeuralNetworkModelSingleLayer, self).__init__()
        if seed is not None:
            torch.manual_seed(seed)
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_nodes),
            nn.ReLU(),
            nn.Linear(hidden_nodes, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

class NeuralNetworkModelDoubleLayers(nn.Module):
    def __init__(self, input_dim, hidden_nodes1=50, hidden_nodes2=100, seed=None):
        super(NeuralNetworkModelDoubleLayers, self).__init__()
        if seed is not None:
            torch.manual_seed(seed)
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_nodes1),
            nn.ReLU(),
            nn.Linear(hidden_nodes1, hidden_nodes2),
            nn.ReLU(),
            nn.Linear(hidden_nodes2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

def train_model(model, X, y, epochs=100, lr=0.01, seed=None):
    """Helper function to train a PyTorch model."""
    if seed is not None:
        torch.manual_seed(seed)

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Convert to tensors
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        output = model(X_tensor)
        loss = criterion(output, y_tensor)
        loss.backward()
        optimizer.step()
    return model

def recourse_loss(model, x):
    prob = model(x)
    prob = prob.view(-1)
    target = torch.ones_like(prob)
    return nn.MSELoss()(prob, target)

### Train Multiple Model with the Same Performance

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
import torch

def compare_and_return_models(d1, d2, f1_type, f2_type, alpha):
    """
    Trains two models on two datasets and returns them if their test accuracies are within alpha.
    """
    X1, y1 = d1
    X2, y2 = d2

    # 80/20 split (fixing seed via random_state)
    X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)
    X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

    def train_and_eval(X_train, y_train, X_test, y_test, model_type):
        input_dim = X_train.shape[1]
        model = model_type(input_dim)
        # Using the globally available train_model function
        train_model(model, X_train, y_train)

        model.eval()
        with torch.no_grad():
            X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
            y_test_tensor = torch.tensor(y_test, dtype=torch.float32)
            probs = model(X_test_tensor).squeeze()
            preds = (probs >= 0.5).float()
            acc = (preds == y_test_tensor).float().mean().item()
        return model, acc

    print("Training Model 1...")
    model1, acc1 = train_and_eval(X1_train, y1_train, X1_test, y1_test, f1_type)
    print(f"Model 1 Test Accuracy: {acc1:.4f}")

    print("Training Model 2...")
    model2, acc2 = train_and_eval(X2_train, y2_train, X2_test, y2_test, f2_type)
    print(f"Model 2 Test Accuracy: {acc2:.4f}")

    diff = abs(acc1 - acc2)
    if diff <= alpha:
        print(f"Success: Accuracy difference {diff:.4f} is within alpha {alpha}.")
        return model1, model2
    else:
        print(f"Error: Test accuracies differ by {diff:.4f}, which exceeds alpha {alpha}.")
        return None, None

def test_compare_and_return_half_datasets_same_models(d, f1_type, f2_type, alpha):
    # read the data
    X, y = d
    num_data = X.shape[0]
    half_data = int(num_data/2)

    # Split into two datasets
    X1, y1 = X[:half_data], y[:half_data]
    X2, y2 = X[half_data:], y[half_data:]
    d1 = (X1, y1)
    d2 = (X2, y2)

    m1, m2 = compare_and_return_models(d1, d2, f1_type, f2_type, alpha)

    if m1 is not None and m2 is not None:
        print("Test passed: Models returned successfully.")
    else:
        print("Test failed: Models were not returned.")

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
import torch

def create_multiplicity_same_model_type(dataset, k, model_class, alpha):
    """
    Divides the dataset into k parts. Trains a model on each part (80% train, 20% test).
    Returns the largest subset of k models whose test accuracies are within alpha of each other.
    Returns None if no subset of size >= 2 is found.
    """
    X, y = dataset
    n_samples = X.shape[0]
    part_size = n_samples // k

    models_stats = []
    print(f"Partitioning dataset ({n_samples} samples) into {k} parts and training models...")

    for i in range(k):
        # 1. Partition the dataset
        start = i * part_size
        # Ensure the last part gets any remainder
        end = (i + 1) * part_size if i < k - 1 else n_samples

        X_part = X[start:end]
        y_part = y[start:end]

        # 2. Split 80/20 for Train/Test
        X_train, X_test, y_train, y_test = train_test_split(
            X_part, y_part, test_size=0.2, random_state=42
        )

        # 3. Train Model
        input_dim = X_train.shape[1]
        model = model_class(input_dim)
        # Assuming train_model is defined in the notebook context
        train_model(model, X_train, y_train, epochs=100, lr=0.01)

        # 4. Compute Accuracy
        model.eval()
        with torch.no_grad():
            X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
            y_test_tensor = torch.tensor(y_test, dtype=torch.float32)
            probs = model(X_test_tensor).squeeze()
            preds = (probs >= 0.5).float()
            acc = (preds == y_test_tensor).float().mean().item()

        models_stats.append({'model': model, 'acc': acc, 'id': i})
        print(f"  Model {i+1}/{k}: Test Accuracy = {acc:.4f}")

    # 5. Find subset within alpha
    # Sort models by accuracy
    models_stats.sort(key=lambda x: x['acc'])

    best_subset = []

    # Sliding window to find the largest subset where (max_acc - min_acc) <= alpha
    left = 0
    for right in range(len(models_stats)):
        # Check if current window [left, right] is valid
        while models_stats[right]['acc'] - models_stats[left]['acc'] > alpha:
            left += 1

        current_window = models_stats[left : right+1]

        # Update best_subset if current is larger
        # Using >= to prefer subsets with higher accuracy if sizes are equal (since list is sorted asc)
        if len(current_window) >= len(best_subset):
            best_subset = current_window

    if len(best_subset) < 2:
        print(f"No subset of models found where accuracies are within alpha {alpha}.")
        return None

    min_acc = best_subset[0]['acc']
    max_acc = best_subset[-1]['acc']
    print(f"Success: Found {len(best_subset)} models with accuracies in [{min_acc:.4f}, {max_acc:.4f}] (Diff: {max_acc-min_acc:.4f} <= {alpha})")

    return [m['model'] for m in best_subset]

# Example Test
# Generate slightly more data for k splits
d_large = get_data_synthetic(2000)
# k=5, so each model gets 400 samples (320 train, 80 test)
subset_models = create_multiplicity_same_model_type(d_large, 5, LogisticRegressionModel, alpha=0.05)

In [ ]:
import sys
import os

import numpy as np
import torch
import random
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim

# --- 1. Rashomon Sampling Logic ---
def get_hessian(model, X, y, lambda_reg=1e-5):
    """
    Computes the Hessian of the Binary Cross Entropy loss w.r.t weights.
    H = (1/N) * X^T * S * X + lambda * I
    where S is diagonal with entries p*(1-p)
    """
    model.eval()
    with torch.no_grad():
        # 1. Forward pass to get probabilities
        # X shape: (N, D)
        X_tensor = torch.tensor(X, dtype=torch.float32)
        probs = model(X_tensor).squeeze() # Shape (N,)

        # 2. Compute diagonal S = p * (1-p)
        S = probs * (1 - probs)

        # 3. Compute X^T * S * X
        # We can do (X.T * S) @ X
        # S is a vector, so we broadcast: X.T * S is equivalent to weighting columns of X.T
        X_weighted = X_tensor.T * S
        H = X_weighted @ X_tensor # (D, D)

        # Normalize by N
        H = H / X.shape[0]

        # Add regularization
        H = H + lambda_reg * torch.eye(H.shape[0])

    return H

def sample_from_ellipsoid(central_theta, H, epsilon, num_samples, seed=42):
    """
    Samples weights theta such that (theta - central)^T H (theta - central) <= 2*epsilon
    """
    rng = np.random.default_rng(seed)
    D = central_theta.shape[0]

    # 1. Compute H^{-1/2} (or Cholesky of Inverse)
    # H = L L^T -> H^-1 = L^-T L^-1
    # We want theta = central + delta
    # delta^T H delta <= 2*epsilon
    # Let delta = H^{-1/2} * z, where ||z||^2 <= 2*epsilon

    # Eigendecomposition is stable
    evals, evecs = torch.linalg.eigh(H)
    # H = U * diag(evals) * U^T
    # H^{-1/2} = U * diag(1/sqrt(evals)) * U^T

    inv_sqrt_evals = 1.0 / torch.sqrt(evals)
    H_inv_sqrt = evecs @ torch.diag(inv_sqrt_evals) @ evecs.T

    samples = []
    for i in range(num_samples):
        # Sample z from ball of radius sqrt(2*epsilon)
        # Direction: Uniform on sphere
        u = rng.standard_normal(D)
        u = u / np.linalg.norm(u)

        # Radius: r = R * (random)^(1/D) for uniformity in volume
        R = np.sqrt(2 * epsilon)
        r = R * (rng.random() ** (1.0/D))

        z = torch.tensor(u * r, dtype=torch.float32)

        # Transform
        delta = H_inv_sqrt @ z
        theta = central_theta + delta
        samples.append(theta)

    return samples

def set_model_weights(model, theta):
    """
    Sets the parameters of the LR model from a flat theta vector.
    Assumes theta = [weights, bias]
    """
    input_dim = model.linear.weight.shape[1]
    with torch.no_grad():
        model.linear.weight.copy_(theta[:input_dim].view(1, -1))
        model.linear.bias.copy_(theta[input_dim].view(-1))

# --- 2. Candidate Selection (Using Rashomon Set) ---
def train_and_select_candidates_rashomon(X, y, max_instances=10, seed=42):
    """
    Trains a central Logistic Regression model using 5-Fold CV, generates a Rashomon set,
    and selects candidate instances. Ensures models are Linear for ElliCE compatibility.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    central_accs = []
    rashomon_avg_accs = []
    final_models = []
    final_candidates = []

    print(f"Starting 5-Fold Cross-Validation (Rashomon Linear) with seed {seed}...")

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        # print(f"  CV Fold {fold+1}/5...")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        input_dim = X.shape[1]
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

        # 1. Train Central Model (Linear)
        central_model = LogisticRegressionModel(input_dim, seed=seed)
        train_model(central_model, X_train, y_train, epochs=500, lr=0.01, seed=seed)

        central_model.eval()
        with torch.no_grad():
            preds = (central_model(X_test_tensor) >= 0.5).float()
            acc = (preds == y_test_tensor).float().mean().item()
        central_accs.append(acc)

        # 2. Generate Rashomon Set
        # Augment X for Hessian (w, b)
        X_train_aug = np.hstack([X_train, np.ones((X_train.shape[0], 1))])

        w = central_model.linear.weight.detach().view(-1)
        b = central_model.linear.bias.detach().view(-1)
        central_theta = torch.cat([w, b])

        probs = central_model(X_train_tensor).squeeze().detach()
        S = probs * (1 - probs)
        X_aug_tensor = torch.tensor(X_train_aug, dtype=torch.float32)
        H = (X_aug_tensor.T * S) @ X_aug_tensor / X_train.shape[0]
        H = H + 1e-5 * torch.eye(H.shape[0])

        # Sample Models
        epsilon = 0.05
        num_models = 4
        # Assumes sample_from_ellipsoid is defined in global scope (Cell 18)
        thetas = sample_from_ellipsoid(central_theta, H, epsilon, num_models, seed=seed+fold)

        current_fold_models = []
        fold_rashomon_accs = []

        for i, theta in enumerate(thetas):
            m = LogisticRegressionModel(input_dim, seed=seed+i)
            set_model_weights(m, theta)
            m.eval()
            with torch.no_grad():
                preds = (m(X_test_tensor) >= 0.5).float()
                acc = (preds == y_test_tensor).float().mean().item()
            fold_rashomon_accs.append(acc)
            current_fold_models.append(m)

        rashomon_avg_accs.append(np.mean(fold_rashomon_accs))

        # 3. Last Fold Candidates
        if fold == 4:
            final_models = current_fold_models
            print("    Identifying candidates in Fold 5 where ALL models predict prob < 0.5...")
            with torch.no_grad():
                all_preds = []
                for model in final_models:
                    preds = model(X_test_tensor).squeeze()
                    all_preds.append(preds)
                all_preds = torch.stack(all_preds)
                is_negative = (all_preds < 0.5).all(dim=0)
                indices = is_negative.nonzero(as_tuple=True)[0]
                for idx in indices:
                    final_candidates.append(X_test_tensor[idx])
            print(f"    Found {len(final_candidates)} candidate instances in the last fold.")

    print(f"  CV Results: Central Acc={np.mean(central_accs):.3f}, Ensemble Acc={np.mean(rashomon_avg_accs):.3f}")

    if len(final_candidates) > max_instances:
        selected_instances = random.sample(final_candidates, max_instances)
    else:
        selected_instances = final_candidates

    return final_models, selected_instances

In [ ]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split
import random

def train_and_select_candidates(X, y, max_instances = 10, seed=42):
    """
    Trains models on the provided dataset and selects candidate instances
    (predicted negative by ALL models) for recourse.

    Args:
        X: Features (numpy array)
        y: Labels (numpy array)
        max_instances: Number of instances to select
        seed: Random seed for reproducibility

    Returns:
        models: List of trained PyTorch models (1 LR, 3 NNs)
        selected_instances: List of candidate instances (tensors)
    """
    # 0. Set global seeds
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # 1. 80/20 Train/Test Split
    print(f"Splitting data 80/20 with seed {seed}...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

    input_dim = X.shape[1]

    # Prepare test tensors for evaluation
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

    # 2. Train Models
    models = []

    # 2a: Logistic Regression
    print("Training Logistic Regression Model...")
    lr_model = LogisticRegressionModel(input_dim, seed=seed)
    train_model(lr_model, X_train, y_train, epochs=50, lr=0.001, seed=seed)

    # Evaluate LR
    lr_model.eval()
    with torch.no_grad():
        preds = (lr_model(X_test_tensor) >= 0.5).float()
        acc = (preds == y_test_tensor).float().mean().item()
    print(f"  Logistic Regression Test Accuracy: {acc:.4f}")

    models.append(lr_model)

    # 2b: 3 Neural Networks
    for i in range(3):
        print(f"Training Neural Network {i+1}...")
        # Use a distinct seed for each NN to ensure diversity in initialization
        current_seed = seed + i + 1
        nn_model = NeuralNetworkModelSingleLayer(input_dim, hidden_nodes=20, seed=current_seed)
        train_model(nn_model, X_train, y_train, epochs=50, lr=0.001, seed=current_seed)

        # Evaluate NN
        nn_model.eval()
        with torch.no_grad():
            preds = (nn_model(X_test_tensor) >= 0.5).float()
            acc = (preds == y_test_tensor).float().mean().item()
        print(f"  Neural Network {i+1} Test Accuracy: {acc:.4f}")

        models.append(nn_model)

    # 3. Find candidates in Test Set (All models prob < 0.5)
    print("Identifying candidates where ALL models predict prob < 0.5...")
    # Set all models to eval mode
    for m in models:
        m.eval()

    candidates = []
    with torch.no_grad():
        # Collect predictions from all models
        all_preds = []
        for model in models:
            preds = model(X_test_tensor).squeeze()
            all_preds.append(preds)

        # Stack to shape [num_models, num_samples]
        all_preds = torch.stack(all_preds)

        # Check if ALL models predict < 0.5
        # (all_preds < 0.5) is a boolean tensor, .all(dim=0) checks across models for each sample
        is_negative = (all_preds < 0.5).all(dim=0)

        # Get indices of samples that satisfy the condition
        indices = is_negative.nonzero(as_tuple=True)[0]

        for idx in indices:
            candidates.append(X_test_tensor[idx])

    print(f"Found {len(candidates)} candidate instances.")

    # 4. Pick max_instances at random
    if len(candidates) > max_instances:
        selected_instances = random.sample(candidates, max_instances)
    else:
        selected_instances = candidates

    return models, selected_instances

In [ ]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split
import random

def train_and_select_candidates_v2(X, y, num_linear=1, num_nonlinear=3, max_instances=10, seed=42):
    """
    Trains a specified number of linear and non-linear models on the dataset
    and selects candidate instances (majority predict negative) for recourse.
    """
    # Set seeds for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    print(f"Splitting data 80/20 with seed {seed}...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

    input_dim = X.shape[1]
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

    models = []

    # 1. Train Linear Models
    for i in range(num_linear):
        current_seed = seed + i
        print(f"Training Logistic Regression Model {i+1}/{num_linear}...")
        lr_model = LogisticRegressionModel(input_dim, seed=current_seed)
        train_model(lr_model, X_train, y_train, epochs=50, lr=0.001, seed=current_seed)

        lr_model.eval()
        with torch.no_grad():
            preds = (lr_model(X_test_tensor) >= 0.5).float()
            acc = (preds == y_test_tensor).float().mean().item()
        print(f"  LR Model {i+1} Test Accuracy: {acc:.4f}")
        models.append(lr_model)

    # 2. Train Non-linear Models
    for i in range(num_nonlinear):
        current_seed = seed + num_linear + i
        print(f"Training Neural Network {i+1}/{num_nonlinear}...")
        nn_model = NeuralNetworkModelSingleLayer(input_dim, hidden_nodes=20, seed=current_seed)
        train_model(nn_model, X_train, y_train, epochs=50, lr=0.001, seed=current_seed)

        nn_model.eval()
        with torch.no_grad():
            preds = (nn_model(X_test_tensor) >= 0.5).float()
            acc = (preds == y_test_tensor).float().mean().item()
        print(f"  NN Model {i+1} Test Accuracy: {acc:.4f}")
        models.append(nn_model)

    # 3. Find candidates in Test Set (Majority vote prob < 0.5)
    print("Identifying candidates where MAJORITY of models predict prob < 0.5...")
    for m in models:
        m.eval()

    candidates = []
    with torch.no_grad():
        all_preds = []
        for model in models:
            all_preds.append(model(X_test_tensor).squeeze())
        
        all_preds = torch.stack(all_preds)
        # Use majority vote: at least half the models predict < 0.5
        is_negative = (all_preds < 0.5).float().mean(dim=0) >= 0.5
        indices = is_negative.nonzero(as_tuple=True)[0]

        for idx in indices:
            candidates.append(X_test_tensor[idx])

    print(f"Found {len(candidates)} candidate instances.")

    # 4. Pick max_instances at random
    if len(candidates) > max_instances:
        selected_instances = random.sample(candidates, max_instances)
    else:
        selected_instances = candidates

    return models, selected_instances

def train_and_select_candidates_v3_rashomon(X, y, num_linear=1, num_nonlinear=3, max_instances=10, seed=42, epsilon=0.05):
    """
    Similar to v2, but linear models are sampled from the Rashomon set (Hessian-based),
    and all models must have an error difference of at most epsilon from the base linear model.
    Will keep training/sampling until the requested number of models is reached.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    print(f"Splitting data 80/20 with seed {seed}...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

    input_dim = X.shape[1]
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

    # 0. Train Base Linear Model
    print("Training Base Logistic Regression Model...")
    base_model = LogisticRegressionModel(input_dim, seed=seed)
    train_model(base_model, X_train, y_train, epochs=500, lr=0.01, seed=seed)

    base_model.eval()
    with torch.no_grad():
        preds = (base_model(X_test_tensor) >= 0.5).float()
        base_acc = (preds == y_test_tensor).float().mean().item()
    print(f"  Base LR Model Test Accuracy: {base_acc:.4f}")

    accepted_models = []

    # 1. Generate Linear Models from Rashomon Set
    if num_linear > 0:
        print(f"Generating {num_linear} Linear Models from Rashomon Set...")
        X_train_aug = np.hstack([X_train, np.ones((X_train.shape[0], 1))])
        w = base_model.linear.weight.detach().view(-1)
        b = base_model.linear.bias.detach().view(-1)
        central_theta = torch.cat([w, b])

        probs = base_model(X_train_tensor).squeeze().detach()
        S = probs * (1 - probs)
        X_aug_tensor = torch.tensor(X_train_aug, dtype=torch.float32)
        H = (X_aug_tensor.T * S) @ X_aug_tensor / X_train.shape[0]
        H = H + 1e-5 * torch.eye(H.shape[0])

        accepted_linear = 0
        attempts = 0
        max_attempts = num_linear * 20
        
        while accepted_linear < num_linear and attempts < max_attempts:
            thetas = sample_from_ellipsoid(central_theta, H, epsilon, 1, seed=seed+attempts)
            theta = thetas[0]
            m = LogisticRegressionModel(input_dim, seed=seed+attempts)
            set_model_weights(m, theta)
            m.eval()
            with torch.no_grad():
                preds = (m(X_test_tensor) >= 0.5).float()
                acc = (preds == y_test_tensor).float().mean().item()

            if base_acc - acc <= epsilon:
                print(f"  Rashomon LR {accepted_linear+1}/{num_linear} Test Accuracy: {acc:.4f} (Diff: {base_acc - acc:.4f})")
                accepted_models.append(m)
                accepted_linear += 1
            attempts += 1
            
        if accepted_linear < num_linear:
            print(f"Warning: Could only find {accepted_linear} linear models within {max_attempts} attempts.")

    # 2. Train Non-linear Models
    print(f"Generating {num_nonlinear} Non-Linear Models...")
    accepted_nonlinear = 0
    attempts = 0
    max_attempts = num_nonlinear * 20
    
    while accepted_nonlinear < num_nonlinear and attempts < max_attempts:
        current_seed = seed + num_linear + attempts + 100
        nn_model = NeuralNetworkModelSingleLayer(input_dim, hidden_nodes=20, seed=current_seed)
        train_model(nn_model, X_train, y_train, epochs=50, lr=0.001, seed=current_seed)

        nn_model.eval()
        with torch.no_grad():
            preds = (nn_model(X_test_tensor) >= 0.5).float()
            acc = (preds == y_test_tensor).float().mean().item()

        if base_acc - acc <= epsilon:
            print(f"  NN Model {accepted_nonlinear+1}/{num_nonlinear} Test Accuracy: {acc:.4f} (Diff: {base_acc - acc:.4f})")
            accepted_models.append(nn_model)
            accepted_nonlinear += 1
        attempts += 1
        
    if accepted_nonlinear < num_nonlinear:
        print(f"Warning: Could only find {accepted_nonlinear} non-linear models within {max_attempts} attempts.")

    print(f"Total accepted models: {len(accepted_models)} / {num_linear + num_nonlinear}")

    if len(accepted_models) == 0:
        print("No models accepted within the epsilon threshold. Falling back to base model.")
        accepted_models = [base_model]

    # 3. Find candidates in Test Set (Majority vote prob < 0.5)
    print("Identifying candidates where MAJORITY of accepted models predict prob < 0.5...")
    for m in accepted_models:
        m.eval()

    candidates = []
    with torch.no_grad():
        all_preds = []
        for model in accepted_models:
            all_preds.append(model(X_test_tensor).squeeze())

        all_preds = torch.stack(all_preds)
        is_negative = (all_preds < 0.5).float().mean(dim=0) >= 0.5
        indices = is_negative.nonzero(as_tuple=True)[0]

        for idx in indices:
            candidates.append(X_test_tensor[idx])

    print(f"Found {len(candidates)} candidate instances.")

    # 4. Pick max_instances at random
    if len(candidates) > max_instances:
        selected_instances = random.sample(candidates, max_instances)
    else:
        selected_instances = candidates

    return accepted_models, selected_instances

### Our Algorithm: Projection, Oracle and the Dynamics

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
# Removed unused import: data_processing.data_loading as dl

def project_l2_ball(x, x0, delta):
    """
    Projects x onto the L2-ball centered at x0 with radius delta.
    Constraint: ||x - x0||_2 <= delta
    """
    diff = x - x0
    norm = torch.norm(diff, p=2)

    # If strictly outside the ball, scale it back to the boundary
    if norm > delta:
        diff = diff * (delta / norm)
        x = x0 + diff
    return x

def oracle_gradient_descent(weights, models, x0, delta, loss_func, lr=0.01, tol=1e-4, num_steps=500, cost_func=None):
    """
    Solves the Oracle problem using Projected Gradient Descent.

    Args:
        weights: Numpy array or Tensor of shape (n,) summing to 1.
        models: List of differentiable PyTorch models.
        x0: Original input (PyTorch Tensor).
        delta: Constraint radius (float).
        loss_func: Function l(model, x) -> differentiable scalar Tensor.
        cost_func: (Optional) Used for verification, but projection handles the constraint.
        lr: Learning rate for the internal optimization.
        num_steps: Maximum number of gradient steps to take.
        tol: Convergence tolerance for the change in x (L2 norm).

    Returns:
        x_opt: The optimized recourse input (PyTorch Tensor).
    """
    # 1. Initialize optimization variable x
    # Start at x0 (or slightly perturbed if needed to break symmetry)
    x = x0.clone().detach().requires_grad_(True)

    # Convert weights to tensor if they aren't already
    if not torch.is_tensor(weights):
        w_tensor = torch.tensor(weights, dtype=torch.float32, device=x0.device)
    else:
        w_tensor = weights.to(x0.device)

    # 2. Define Optimizer (SGD or Adam)
    optimizer = optim.Adam([x], lr=lr)

    # Track previous x for convergence check
    prev_x = x.clone().detach()

    for step in range(num_steps):
        optimizer.zero_grad()

        # 3. Compute Weighted Loss
        # L(x) = sum( w_i * loss(h_i, x) )
        # Initialize total_loss as 0.0 to avoid in-place operations on leaf variables
        total_loss = 0.0
        for i, model in enumerate(models):
            loss_i = loss_func(model, x)
            # Use standard addition, not in-place (+=)
            total_loss = total_loss + w_tensor[i] * loss_i

        # 4. Backpropagation
        total_loss.backward()

        # 5. Gradient Step
        optimizer.step()

        # 6. Projection Step (Enforce Constraint)
        # We manually project the data of x back into the feasible set
        with torch.no_grad():
            x.data = project_l2_ball(x.data, x0, delta)

            # Convergence Check
            if tol is not None:
                dist = torch.norm(x.data - prev_x, p=2).item()
                if dist < tol:
                    # Converged
                    break
                prev_x = x.clone().detach()

    return x.detach()

def solve_robust_recourse_with_oracle(x0, models, delta, loss_func, oracle_func, T=100, eta=None, cost_func=None, B=1):
    """
    Solves the Robust Recourse problem using an Oracle-Efficient Minimax approach.

    Args:
        x0: The original input instance (e.g., numpy array or tensor).
        models: A list of n classifiers [h_1, ..., h_n].
        delta: The float constraint budget for the cost function.
        loss_func: A function l(model, x) -> float returning the loss.
        cost_func: A function c(x_original, x_new) -> float (used by the Oracle).
        oracle_func: A function Oracle(weights, models, x0, delta, loss, cost) -> x_best.
                     It solves min_x sum(w_i * loss_i(x)) s.t. cost <= delta.
        T: Integer number of iterations.
        eta: Float learning rate for the adversary. If None, set theoretically.
        B: Upper bound on the loss function value, used for theoretical learning rate.

    Returns:
        x_bar: The average robust recourse found over T rounds.
        final_weights: The final probability distribution over models.
        x_bar_history: List of average robust recourse vectors at each step t=1...T.
        weight_history: List of weight vectors at each step t=1...T.
        eta_losses_history: List of (eta * losses) vectors at each step t=1...T.
    """
    n = len(models)

    # 1. Initialize Adversary Weights (Uniform Distribution)
    w = np.ones(n) / n

    # Set theoretical learning rate if not provided
    # eta = (1/B) * sqrt(ln(n) / T)
    if eta is None:
        eta = (1.0 / B) * np.sqrt(np.log(n) / T)

    # Storage for the cumulative sum of x (for averaging)
    x_sum = None
    x_bar_history = []
    weight_history = []
    eta_losses_history = []

    for t in range(T):
        # Record weights at the start of the round
        weight_history.append(w.copy())

        # --- Step 1: Agent Best Response (Oracle Call) ---
        # The Agent finds the optimal x_t against the current weighted mixture w_t
        x_t = oracle_func(w, models, x0, delta, loss_func)

        # Initialize accumulator on first iteration
        if x_sum is None:
            x_sum = torch.zeros_like(x_t)
        x_sum += x_t

        # Store average so far
        # We need to detach/clone to avoid storing the graph or reference to changing variable
        current_x_bar = (x_sum / (t + 1)).clone().detach()
        x_bar_history.append(current_x_bar)

        # --- Step 2: Adversary Update (Multiplicative Weights) ---
        # Calculate the loss vector for the chosen x_t against all models
        # Use .item() to detach and get scalar
        losses = np.array([loss_func(h, x_t).item() for h in models])

        # Calculate the update term
        # In the maximization game, w_{t+1} = w_t * exp(eta * losses)
        update_term = eta * losses
        eta_losses_history.append(update_term)

        # Update weights
        w = w * np.exp(update_term)

        # Normalize weights to stay on the simplex
        w = w / np.sum(w)

    # --- Return Time-Averaged Solution ---
    x_bar = x_sum / T

    return x_bar, w, x_bar_history, weight_history, eta_losses_history

## Experiment 1: Convergence


In [ ]:
#Convergence Analysis
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
import data_processing.data_loading as dl

print("Starting Advanced Convergence Experiment...")

# --- 1. Configuration ---
max_instances = 20
seed = 42
T = 100
folds = 2

# Sets of parameters to sweep
budgets = [2.0, 5.0]
model_configs = [
    {'name': '4 LR', 'num_linear': 4, 'num_nonlinear': 0},
    {'name': '1 LR + 3 NN', 'num_linear': 1, 'num_nonlinear': 3},
    {'name': '2 LR + 6 NN', 'num_linear': 2, 'num_nonlinear': 6}
]

# Load Datasets safely
datasets = {}
try:
    # Using 20,000 for synthetic data based on previous baselines
    X_syn, y_syn = get_data_synthetic(20000)
    datasets['Synthetic'] = (X_syn, y_syn)
except Exception as e:
    print(f"Skipping Synthetic: {e}")

try:
    X_com, y_com = dl.load_compas_notebook()
    datasets['COMPAS'] = (X_com, y_com)
except Exception as e:
    print(f"Skipping COMPAS: {e}")

try:
    X_pol, y_pol = dl.load_polish_notebook()
    datasets['Polish'] = (X_pol, y_pol)
except Exception as e:
    print(f"Skipping Polish: {e}")

try:
    # Using load_germanc as identified from the module
    X_ger, y_ger = dl.load_germanc_notebook()
    datasets['German'] = (X_ger, y_ger)
except Exception as e:
    print(f"Skipping German: {e}")

all_trajectories = []

# --- 2. Run Experiment ---
for ds_name, (X, y) in datasets.items():
    print(f"\n--- Processing Dataset: {ds_name} ---")
    scaler = StandardScaler()
    X_norm = scaler.fit_transform(X)

    for config in model_configs:
        for budget in budgets:
            print(f"  Config: {config['name']}, Budget: {budget}")
            all_instance_trajectories = []

            for fold in range(folds):
                current_seed = seed + fold
                try:
                    models, candidates = train_and_select_candidates_v2(
                        X_norm, y,
                        num_linear=config['num_linear'],
                        num_nonlinear=config['num_nonlinear'],
                        max_instances=max_instances,
                        seed=current_seed
                    )
                except Exception as e:
                    print(f"    Fold {fold+1} error: {e}")
                    continue

                if not candidates:
                    print("    No candidates found.")
                    continue

                for x0 in candidates:
                    # Run ORPM (Oracle Dynamics)
                    _, _, x_bar_hist, _, _ = solve_robust_recourse_with_oracle(
                        x0, models, delta=budget, loss_func=recourse_loss,
                        oracle_func=oracle_gradient_descent, T=T, B=1
                    )

                    # Compute Max Loss curve for this instance
                    traj = []
                    for x_val in x_bar_hist:
                        losses = [recourse_loss(m, x_val).item() for m in models]
                        traj.append(max(losses))
                    all_instance_trajectories.append(traj)

            if all_instance_trajectories:
                # Average over all instances across all folds and calculate standard deviation
                mean_curve = np.mean(all_instance_trajectories, axis=0)
                std_curve = np.std(all_instance_trajectories, axis=0)
                all_trajectories.append({
                    'dataset': ds_name,
                    'config': config['name'],
                    'budget': budget,
                    'mean': mean_curve,
                    'std': std_curve,
                    'trajectory': mean_curve # Keep for backward compatibility if needed
                })

# --- 3. Plotting ---
print("\nPlotting Loss Trajectories...")
plt.figure(figsize=(14, 8))

# Define some distinct styles for clarity
colors = plt.cm.tab10.colors
linestyles = ['-', '--', ':', '-.']

for idx, data in enumerate(all_trajectories):
    label = f"{data['dataset']} | {data['config']} | Budget={data['budget']}"
    c = colors[idx % len(colors)]
    ls = linestyles[(idx // len(colors)) % len(linestyles)]

    plt.plot(range(1, T+1), data['mean'], label=label, linewidth=2, color=c, linestyle=ls)
    plt.fill_between(range(1, T+1), data['mean'] - data['std'], data['mean'] + data['std'], color=c, alpha=0.2)

plt.xlabel('Iteration (T)')
plt.ylabel('Average Max Recourse Loss')
plt.title('Convergence Analysis: Loss Trajectory over Time (Oracle Dynamics)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_selected_trajectories(data_list, selected_triples):
    """
    Plots a subset of trajectories based on the selected (dataset, config, budget) triples.
    Includes variance (shaded region) if available.
    
    Args:
        data_list: List of dictionaries containing the trajectory data. 
                   Should have 'dataset', 'config', 'budget', and 
                   ideally 'mean' and 'std' (or 'fold_trajectories').
        selected_triples: List of tuples (dataset_name, config_name, budget)
                          e.g., [('Synthetic', '4 LR', 2.0), ('COMPAS', '1 LR + 3 NN', 5.0)]
    """
    plt.figure(figsize=(12, 8))
    colors = plt.cm.tab10.colors
    linestyles = ['-', '--', ':', '-.']
    
    plot_idx = 0
    for data in data_list:
        triple = (data.get('dataset'), data.get('config'), data.get('budget'))
        
        if triple in selected_triples:
            # Determine mean and standard deviation for the shaded variance area
            if 'mean' in data and 'std' in data:
                mean_curve = np.array(data['mean'])
                std_curve = np.array(data['std'])
            elif 'fold_trajectories' in data:
                fold_traj = np.array(data['fold_trajectories'])
                mean_curve = np.mean(fold_traj, axis=0)
                std_curve = np.std(fold_traj, axis=0)
            elif 'trajectory' in data:
                # Fallback if only the mean was stored previously
                mean_curve = np.array(data['trajectory'])
                std_curve = np.zeros_like(mean_curve)
                print(f"Warning: No variance data found for {triple}. Plotting mean only.")
            else:
                continue
                
            label = f"{triple[0]} | {triple[1]} | B={triple[2]}"
            c = colors[plot_idx % len(colors)]
            ls = linestyles[(plot_idx // len(colors)) % len(linestyles)]
            steps = range(1, len(mean_curve) + 1)
            
            # Plot Mean Curve
            plt.plot(steps, mean_curve, label=label, linewidth=2, color=c, linestyle=ls)
            
            # Plot Variance (Shaded Region)
            if np.any(std_curve > 0):
                plt.fill_between(steps, mean_curve - std_curve, mean_curve + std_curve, color=c, alpha=0.2)
                
            plot_idx += 1
            
    plt.xlabel('Iteration (T)')
    plt.ylabel('Average Max Recourse Loss')
    plt.title('Convergence Analysis: Selected Loss Trajectories')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


#Define the specific triples you want to plot
my_subset = [
     ('Synthetic', '2 LR + 6 NN', 5.0),
     ('COMPAS', '2 LR + 6 NN', 5.0),
     ('German', '4 LR', 5.0),
     ('Polish', '4 LR', 5.0),
 ]
plot_selected_trajectories(all_trajectories, my_subset)

## Experiment 2: Comparison with Baselines ##


In [ ]:
def ADV(x0, models, B, loss_func, T=100, lr=0.01, tol=1e-4):
    """
    ADV (Adversarial Training Baseline for Algorithmic Recourse).
    Optimizes x to minimize the WORST-CASE loss among the FINITE SET of models (Minimax).

    Args:
        x0: Original instance.
        models: List of PyTorch models.
        B: Constraint radius (recourse budget).
        loss_func: Loss function l(model, x).
        T: Maximum number of iterations.
        lr: Learning rate.
        tol: Threshold for change in loss to trigger early stopping.
    """
    # Initialize x at x0
    x = x0.clone().detach().requires_grad_(True)
    history = []

    # Optimizer
    optimizer = torch.optim.Adam([x], lr=lr)

    prev_loss = float('inf')

    for t in range(T):
        optimizer.zero_grad()

        # Compute losses for all models
        losses = []
        for model in models:
            losses.append(loss_func(model, x))

        # Stack losses into a tensor to find the max
        losses_tensor = torch.stack(losses)

        # Identify the worst-case loss (Max)
        max_loss = torch.max(losses_tensor)

        # Early Stopping Check
        current_loss = max_loss.item()
        if abs(prev_loss - current_loss) < tol:
            break
        prev_loss = current_loss

        # Backward pass on the max loss
        max_loss.backward()
        optimizer.step()

        # Projection Step (Constraint: ||x - x0|| <= B)
        with torch.no_grad():
            x.data = project_l2_ball(x.data, x0, B)

        # Store current state
        history.append(x.clone().detach())

    return history

def ROAR(x0, initial_model, delta_model, B, loss_func, T=100, lr=0.01, tol=1e-4):
    x = x0.clone().detach().requires_grad_(True)
    history = []
    optimizer = torch.optim.Adam([x], lr=lr)

    w_init = initial_model.linear.weight.detach().view(-1)
    b_init = initial_model.linear.bias.detach().view(-1)
    device = x.device

    prev_loss = float('inf')

    for t in range(T):
        optimizer.zero_grad()
        x_sign = torch.sign(x.data)
        w_adv = w_init - delta_model * x_sign
        b_adv = b_init - delta_model

        logit = torch.dot(w_adv, x) + b_adv
        prob = torch.sigmoid(logit).view(-1) # Ensure 1D

        # FIX: Make target same shape as prob
        target = torch.ones_like(prob)

        current_loss = nn.MSELoss()(prob, target)

        loss_val = current_loss.item()
        if abs(prev_loss - loss_val) < tol:
            break
        prev_loss = loss_val

        current_loss.backward()
        optimizer.step()

        with torch.no_grad():
            x.data = project_l2_ball(x.data, x0, B)

        history.append(x.clone().detach())

    return history

In [ ]:
def ElliCE(x0, models, B, loss_func, T=100, lr=0.01, tol=1e-4):
    # Re-calculate Sigma and omega_c logic as before
    thetas = []
    device = x0.device
    for m in models:
        w = m.linear.weight.detach().view(-1)
        b = m.linear.bias.detach().view(-1)
        theta = torch.cat([w, b])
        thetas.append(theta)
    thetas = torch.stack(thetas).to(device)
    omega_c = torch.mean(thetas, dim=0)
    diff = thetas - omega_c
    N = thetas.shape[0]
    if N > 1:
        Sigma = (diff.T @ diff) / (N - 1)
    else:
        Sigma = torch.eye(len(omega_c)).to(device) * 1e-4
    Sigma = Sigma + torch.eye(len(omega_c)).to(device) * 1e-5

    try:
        Sigma_inv = torch.linalg.inv(Sigma)
    except:
        Sigma_inv = torch.eye(len(omega_c)).to(device)

    dists = []
    for i in range(N):
        d = diff[i]
        dist_sq = (d.T @ Sigma_inv @ d)
        dists.append(dist_sq)

    if N > 1:
        epsilon = max(dists).item() / 2.0
        epsilon *= 1.1
    else:
        epsilon = 0.1

    x = x0.clone().detach().requires_grad_(True)
    history = []
    optimizer = torch.optim.Adam([x], lr=lr)

    prev_loss = float('inf')

    for t in range(T):
        optimizer.zero_grad()
        h_tilde = torch.cat([x, torch.tensor([1.0]).to(device)])
        denom = h_tilde.T @ Sigma @ h_tilde
        if denom < 1e-8: denom = 1e-8
        scale = torch.sqrt(2 * epsilon / denom)
        theta_w = omega_c - scale * (Sigma @ h_tilde)

        logit = torch.dot(theta_w, h_tilde)
        prob = torch.sigmoid(logit).view(-1) # Ensure 1D

        # FIX: Make target same shape as prob
        target = torch.ones_like(prob)

        current_loss = nn.MSELoss()(prob, target)

        loss_val = current_loss.item()
        if abs(prev_loss - loss_val) < tol:
            break
        prev_loss = loss_val

        current_loss.backward()
        optimizer.step()

        with torch.no_grad():
            x.data = project_l2_ball(x.data, x0, B)

        history.append(x.clone().detach())

    return history

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
import random
import data_processing.data_loading as dl

def comparison_with_baselines_linear(dataset_name, dataset, recourse_budget=5.0, folds=5, max_instances=5, seed=42, T=200, lr=0.01):
    """
    Compares ORPM, ADV, ElliCE, and ROAR on Robust Validity.
    """
    print(f"\n=== Running Comparison on {dataset_name} (Budget={recourse_budget}) ===")
    X, y = dataset

    # Normalize Data
    scaler = StandardScaler()
    X_norm = scaler.fit_transform(X)

    # Storage for results
    results_validity = {
        'ORPM': [],
        'ADV': [],
        'ElliCE': [],
        'ROAR': []
    }
    results_loss = {
        'ORPM': [],
        'ADV': [],
        'ElliCE': [],
        'ROAR': []
    }

    for fold in range(folds):
        print(f"  Experiment Repetition {fold+1}/{folds}...")
        current_seed = seed + fold

        # 1. Train Models & Select Candidates (Rashomon Set)
        try:
            models, candidates = train_and_select_candidates_rashomon(X_norm, y, max_instances=max_instances, seed=current_seed)
        except Exception as e:
            print(f"    Error in fold {fold+1}: {e}")
            continue

        if not candidates:
            print("    No candidates found.")
            continue

        # --- Prepare ROAR inputs (L-inf Delta) ---
        initial_model = models[0]
        delta_model = 0.0
        with torch.no_grad():
            w_init = initial_model.linear.weight.detach()
            b_init = initial_model.linear.bias.detach()
            for m in models:
                w = m.linear.weight.detach()
                b = m.linear.bias.detach()
                diff_w = torch.abs(w - w_init)
                diff_b = torch.abs(b - b_init)
                max_diff = max(torch.max(diff_w).item(), diff_b.item())
                if max_diff > delta_model:
                    delta_model = max_diff
        print(f"    ROAR Parameters: L-inf Delta = {delta_model:.6f}")

        # Track metrics for this fold
        fold_validity = {'ORPM': 0, 'ADV': 0, 'ElliCE': 0, 'ROAR': 0}
        fold_loss = {'ORPM': 0.0, 'ADV': 0.0, 'ElliCE': 0.0, 'ROAR': 0.0}
        total = len(candidates)

        for x0 in candidates:
            # --- Helper to evaluate result ---
            def evaluate_recourse(x_rec, models, method_name):
                valid = True
                losses = []
                with torch.no_grad():
                    for m in models:
                        prob = m(x_rec).view(-1).item()
                        loss = recourse_loss(m, x_rec).item()
                        losses.append(loss)
                        if prob < 0.5:
                            valid = False
                max_loss = max(losses)
                fold_loss[method_name] += max_loss
                if valid: fold_validity[method_name] += 1

            # --- A. ORPM ---
            x_orpm, _, _, _, _ = solve_robust_recourse_with_oracle(
                x0, models, recourse_budget, recourse_loss, oracle_gradient_descent, T=T, B=1
            )
            evaluate_recourse(x_orpm, models, 'ORPM')

            # --- B. ADV ---
            hist_adv = ADV(x0, models, B=recourse_budget, loss_func=recourse_loss, T=T, lr=lr)
            x_adv = hist_adv[-1]
            evaluate_recourse(x_adv, models, 'ADV')

            # --- C. ElliCE ---
            hist_ellice = ElliCE(x0, models, B=recourse_budget, loss_func=recourse_loss, T=T, lr=lr)
            x_ellice = hist_ellice[-1]
            evaluate_recourse(x_ellice, models, 'ElliCE')

            # --- D. ROAR ---
            hist_roar = ROAR(x0, initial_model, delta_model, B=recourse_budget, loss_func=recourse_loss, T=T, lr=lr)
            x_roar = hist_roar[-1]
            evaluate_recourse(x_roar, models, 'ROAR')

        # Calculate Fold Averages
        if total > 0:
            for method in ['ORPM', 'ADV', 'ElliCE', 'ROAR']:
                results_validity[method].append(fold_validity[method] / total)
                results_loss[method].append(fold_loss[method] / total)

    # Aggregate Results
    summary = {}
    for method in ['ORPM', 'ADV', 'ElliCE', 'ROAR']:
        if results_validity[method]:
            v_mean = np.mean(results_validity[method]) * 100
            v_std = np.std(results_validity[method]) * 100
            l_mean = np.mean(results_loss[method])
            l_std = np.std(results_loss[method])
            summary[f"{method} Validity"] = f"{v_mean:.2f}% (+/- {v_std:.2f})"
            summary[f"{method} Loss"] = f"{l_mean:.4f} (+/- {l_std:.4f})"
        else:
            summary[f"{method} Validity"] = "N/A"
            summary[f"{method} Loss"] = "N/A"

    return pd.DataFrame(summary, index=[dataset_name])

In [ ]:
import numpy as np
import random
import pandas as pd
import data_processing.data_loading as dl

# --- Execution of Comparison ---
print("Starting full comparison experiment (Re-run with ADV, ROAR, and ElliCE)...")
all_results = []

# Configuration
SEEDS = 42
FOLDS = 5
INSTANCES = 200
BUDGET = 5.0
T_STEPS = 200

# 1. Synthetic
print("Loading Synthetic...")
X_syn, y_syn = get_data_synthetic(20000)
res_syn = comparison_with_baselines_linear("Synthetic", (X_syn, y_syn), recourse_budget=BUDGET, folds=FOLDS, max_instances=INSTANCES, seed=SEEDS, T=T_STEPS)
all_results.append(res_syn)

# 2. Polish
try:
    print("Loading Polish...")
    X_pol, y_pol = load('polish')
    res_pol = comparison_with_baselines_linear("Polish", (X_pol, y_pol), recourse_budget=BUDGET, folds=FOLDS, max_instances=INSTANCES, seed=SEEDS, T=T_STEPS)
    all_results.append(res_pol)
except Exception as e:
    print(f"Skipping Polish: {e}")

# 3. COMPAS
try:
    print("Loading COMPAS...")
    X_com, y_com = load('compas')
    res_com = comparison_with_baselines_linear("COMPAS", (X_com, y_com), recourse_budget=BUDGET, folds=FOLDS, max_instances=INSTANCES, seed=SEEDS, T=T_STEPS)
    all_results.append(res_com)
except Exception as e:
    print(f"Skipping COMPAS: {e}")

# --- Separate and Display Tables ---
if all_results:
    final_table = pd.concat(all_results)

    # Split columns based on keyword
    validity_cols = [c for c in final_table.columns if "Validity" in c]
    loss_cols = [c for c in final_table.columns if "Loss" in c]

    print("\n=== Robust Validity (% Accepted by All Models) ===")
    display(final_table[validity_cols])

    print("\n=== Average Robust Loss (Minimizing Worst-Case) ===")
    display(final_table[loss_cols])
else:
    print("No results generated.")

In [ ]:
import lime
import lime.lime_tabular
import numpy as np
import torch

def linearize_models_lime(models, x0, X_train):
    """
    Generates local linear surrogates for a list of models using LIME centered at x0.

    Args:
        models: List of PyTorch models (classifiers).
        x0: Numpy array representing the instance to explain.
        X_train: Numpy array of training data (used for LIME statistics).

    Returns:
        List of LogisticRegressionModel objects initialized with LIME coefficients.
    """
    input_dim = X_train.shape[1]

    # Initialize LIME explainer
    # discretize_continuous=False ensures we get coefficients for the continuous features directly
    explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train,
        mode='classification',
        discretize_continuous=False,
        verbose=False
    )

    surrogates = []

    for model in models:
        # LIME expects a function that takes numpy array (n_samples, n_features)
        # and returns (n_samples, n_classes) probabilities.
        def predict_proba_fn(X_numpy):
            # Convert numpy array to PyTorch tensor
            X_tensor = torch.tensor(X_numpy, dtype=torch.float32)

            # Ensure model is in eval mode
            model.eval()
            with torch.no_grad():
                # Model outputs probability of class 1 (sigmoid output)
                prob_1 = model(X_tensor).numpy().flatten()
                # Calculate probability of class 0
                prob_0 = 1 - prob_1
                # Stack to get (N, 2) array as expected by LIME
                return np.vstack((prob_0, prob_1)).T

        # Generate explanation for x0
        # num_features=input_dim ensures we get weights for all features, not just top K
        exp = explainer.explain_instance(
            data_row=x0,
            predict_fn=predict_proba_fn,
            num_features=input_dim
        )

        # Extract weights and intercept for class 1
        # local_exp is a dictionary: {class_label: [(feature_idx, weight), ...]}
        local_exp = exp.local_exp[1]
        intercept = exp.intercept[1]

        # Construct full weight vector
        w = np.zeros(input_dim)
        for feature_idx, weight in local_exp:
            w[feature_idx] = weight

        # Create a new surrogate LogisticRegressionModel
        surrogate = LogisticRegressionModel(input_dim)

        # Manually set the weights and bias of the surrogate
        with torch.no_grad():
            surrogate.linear.weight.copy_(torch.tensor(w, dtype=torch.float32).view(1, -1))
            surrogate.linear.bias.copy_(torch.tensor(intercept, dtype=torch.float32))

        surrogates.append(surrogate)

    return surrogates

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
import random

def comparison_with_baselines_nonlinear(dataset_name, dataset, recourse_budget=2.0, folds=2, max_instances=5, seed=42, T=100):
    """
    Compares ORPM and ADV (on original models) vs ElliCE and ROAR (on LIME surrogates).
    Evaluates all resulting recourses on the ORIGINAL models.
    """
    print(f"\n=== Running Non-Linear Comparison on {dataset_name} (Budget={recourse_budget}) ===")
    X, y = dataset

    # Normalize Data
    scaler = StandardScaler()
    X_norm = scaler.fit_transform(X)

    # Storage for results
    results_validity = {
        'ORPM': [],
        'ADV': [],
        'ElliCE': [],
        'ROAR': []
    }
    results_loss = {
        'ORPM': [],
        'ADV': [],
        'ElliCE': [],
        'ROAR': []
    }

    for fold in range(folds):
        print(f"  Experiment Repetition {fold+1}/{folds}...")
        current_seed = seed + fold

        # 1. Train Models (Mixed: LR + NNs) & Select Candidates
        # We reuse the function from Experiment 1 which trains 1 LR and 3 NNs
        try:
            models, candidates = train_and_select_candidates(X_norm, y, max_instances=max_instances, seed=current_seed)
        except Exception as e:
            print(f"    Error in fold {fold+1}: {e}")
            continue

        if not candidates:
            print("    No candidates found.")
            continue

        # Track metrics for this fold
        fold_validity = {'ORPM': 0, 'ADV': 0, 'ElliCE': 0, 'ROAR': 0}
        fold_loss = {'ORPM': 0.0, 'ADV': 0.0, 'ElliCE': 0.0, 'ROAR': 0.0}
        total = len(candidates)

        # We need X_train to pass to LIME. In train_and_select_candidates, split is 80/20.
        # Re-create split to get X_train.
        X_train, _, _, _ = train_test_split(X_norm, y, test_size=0.2, random_state=current_seed)

        for x0 in candidates:
            # --- A. Generate Surrogates using LIME ---
            # x0 is a tensor, convert to numpy for LIME
            x0_np = x0.numpy()
            surrogate_models = linearize_models_lime(models, x0_np, X_train)

            # --- B. Calculate ROAR delta_model on Surrogates ---
            initial_surrogate = surrogate_models[0]
            delta_model = 0.0
            with torch.no_grad():
                w_init = initial_surrogate.linear.weight.detach()
                b_init = initial_surrogate.linear.bias.detach()
                for m in surrogate_models:
                    w = m.linear.weight.detach()
                    b = m.linear.bias.detach()
                    diff_w = torch.abs(w - w_init)
                    diff_b = torch.abs(b - b_init)
                    max_diff = max(torch.max(diff_w).item(), diff_b.item())
                    if max_diff > delta_model:
                        delta_model = max_diff
            # print(f"    LIME Surrogates ROAR Delta: {delta_model:.6f}")

            # --- Helper to evaluate result on ORIGINAL models ---
            def evaluate_recourse(x_rec, original_models, method_name):
                valid = True
                losses = []
                with torch.no_grad():
                    for m in original_models:
                        prob = m(x_rec).view(-1).item()
                        loss = recourse_loss(m, x_rec).item()
                        losses.append(loss)
                        if prob < 0.5:
                            valid = False
                max_loss = max(losses)
                fold_loss[method_name] += max_loss
                if valid: fold_validity[method_name] += 1

            # --- C. Run Methods ---

            # 1. ORPM (on ORIGINAL models)
            x_orpm, _, _, _, _ = solve_robust_recourse_with_oracle(
                x0, models, recourse_budget, recourse_loss, oracle_gradient_descent, T=T, B=1
            )
            evaluate_recourse(x_orpm, models, 'ORPM')

            # 2. ADV (on ORIGINAL models)
            # Optimization directly against the non-linear models
            hist_adv = ADV(x0, models, B=recourse_budget, loss_func=recourse_loss, T=T, lr=0.01)
            x_adv = hist_adv[-1]
            evaluate_recourse(x_adv, models, 'ADV')

            # 3. ElliCE (on SURROGATE models)
            # optimize x against surrogates, evaluate on original
            hist_ellice = ElliCE(x0, surrogate_models, B=recourse_budget, loss_func=recourse_loss, T=T, lr=0.01)
            x_ellice = hist_ellice[-1]
            evaluate_recourse(x_ellice, models, 'ElliCE')

            # 4. ROAR (on SURROGATE models)
            # optimize x against surrogates, evaluate on original
            hist_roar = ROAR(x0, initial_surrogate, delta_model, B=recourse_budget, loss_func=recourse_loss, T=T, lr=0.01)
            x_roar = hist_roar[-1]
            evaluate_recourse(x_roar, models, 'ROAR')

        # Calculate Fold Averages
        if total > 0:
            for method in ['ORPM', 'ADV', 'ElliCE', 'ROAR']:
                results_validity[method].append(fold_validity[method] / total)
                results_loss[method].append(fold_loss[method] / total)

    # Aggregate Results
    summary = {}
    for method in ['ORPM', 'ADV', 'ElliCE', 'ROAR']:
        if results_validity[method]:
            v_mean = np.mean(results_validity[method]) * 100
            v_std = np.std(results_validity[method]) * 100
            l_mean = np.mean(results_loss[method])
            l_std = np.std(results_loss[method])
            summary[f"{method} Validity"] = f"{v_mean:.2f}% (+/- {v_std:.2f})"
            summary[f"{method} Loss"] = f"{l_mean:.4f} (+/- {l_std:.4f})"
        else:
            summary[f"{method} Validity"] = "N/A"
            summary[f"{method} Loss"] = "N/A"

    return pd.DataFrame(summary, index=[dataset_name])

In [ ]:
import time
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

def comparison_with_baselines_nonlinear_v2(dataset_name, dataset, num_linear=1, num_nonlinear=3, recourse_budget=2.0, folds=1, max_instances=5, seed=42, T=100):
    """
    Scalable version: Compares ORPM, ADV, ElliCE, and ROAR on varying number of models.
    Tracks both robust validity and average execution time per instance, including variance.
    """
    print(f"\n=== Running Scalable Comparison on {dataset_name} (Models: {num_linear} LR + {num_nonlinear} NN) ===")
    X, y = dataset
    scaler = StandardScaler()
    X_norm = scaler.fit_transform(X)

    results_validity = {'ORPM': [], 'ADV': [], 'ElliCE': [], 'ROAR': []}
    results_time = {'ORPM': [], 'ADV': [], 'ElliCE': [], 'ROAR': []}

    for fold in range(folds):
        current_seed = seed + fold
        try:
            models, candidates = train_and_select_candidates_v2(X_norm, y, num_linear=num_linear, num_nonlinear=num_nonlinear, max_instances=max_instances, seed=current_seed)
        except Exception as e:
            print(f"    Error in fold {fold+1}: {e}")
            continue

        if not candidates:
            continue

        fold_validity = {'ORPM': 0, 'ADV': 0, 'ElliCE': 0, 'ROAR': 0}
        fold_time = {'ORPM': 0.0, 'ADV': 0.0, 'ElliCE': 0.0, 'ROAR': 0.0}
        total = len(candidates)

        X_train, _, _, _ = train_test_split(X_norm, y, test_size=0.2, random_state=current_seed)

        for x0 in candidates:
            t0 = time.time()
            x0_np = x0.numpy()
            surrogate_models = linearize_models_lime(models, x0_np, X_train)
            lime_overhead = time.time() - t0

            initial_surrogate = surrogate_models[0]
            delta_model = 0.0
            with torch.no_grad():
                w_init = initial_surrogate.linear.weight.detach()
                b_init = initial_surrogate.linear.bias.detach()
                for m in surrogate_models:
                    w = m.linear.weight.detach()
                    b = m.linear.bias.detach()
                    max_diff = max(torch.max(torch.abs(w - w_init)).item(), torch.abs(b - b_init).item())
                    if max_diff > delta_model:
                        delta_model = max_diff

            def evaluate_recourse(x_rec, original_models, method_name, exec_time):
                valid = True
                with torch.no_grad():
                    for m in original_models:
                        if m(x_rec).view(-1).item() < 0.5:
                            valid = False
                fold_time[method_name] += exec_time
                if valid: fold_validity[method_name] += 1

            t0 = time.time()
            x_orpm, _, _, _, _ = solve_robust_recourse_with_oracle(x0, models, recourse_budget, recourse_loss, oracle_gradient_descent, T=T, B=1)
            evaluate_recourse(x_orpm, models, 'ORPM', time.time() - t0)

            t0 = time.time()
            hist_adv = ADV(x0, models, B=recourse_budget, loss_func=recourse_loss, T=T, lr=0.01)
            evaluate_recourse(hist_adv[-1], models, 'ADV', time.time() - t0)

            t0 = time.time()
            hist_ellice = ElliCE(x0, surrogate_models, B=recourse_budget, loss_func=recourse_loss, T=T, lr=0.01)
            evaluate_recourse(hist_ellice[-1], models, 'ElliCE', lime_overhead + (time.time() - t0))

            t0 = time.time()
            hist_roar = ROAR(x0, initial_surrogate, delta_model, B=recourse_budget, loss_func=recourse_loss, T=T, lr=0.01)
            evaluate_recourse(hist_roar[-1], models, 'ROAR', lime_overhead + (time.time() - t0))

        if total > 0:
            for method in ['ORPM', 'ADV', 'ElliCE', 'ROAR']:
                results_validity[method].append(fold_validity[method] / total)
                results_time[method].append(fold_time[method] / total)

    summary = {}
    for method in ['ORPM', 'ADV', 'ElliCE', 'ROAR']:
        if results_validity[method]:
            summary[f"{method} Validity Mean"] = np.mean(results_validity[method]) * 100
            summary[f"{method} Validity Std"] = np.std(results_validity[method]) * 100
            summary[f"{method} Time Mean (s)"] = np.mean(results_time[method])
            summary[f"{method} Time Std (s)"] = np.std(results_time[method])
        else:
            summary[f"{method} Validity Mean"] = np.nan
            summary[f"{method} Validity Std"] = np.nan
            summary[f"{method} Time Mean (s)"] = np.nan
            summary[f"{method} Time Std (s)"] = np.nan

    return pd.DataFrame(summary, index=[f"{num_linear+num_nonlinear} Models ({num_linear}LR+{num_nonlinear}NN)"])

In [ ]:
#Results for Tradeoff Between Validity and Budget and also Size of the Competing Set
import torch
import torch.nn as nn
import data_processing.data_loading as dl

all_results_nonlinear = []

# Configuration
SEEDS = 42
FOLDS = 5
INSTANCES = 200
BUDGET = 5.0
T_STEPS = 200


# 1. Synthetic
print("Processing Synthetic...")
X_syn, y_syn = get_data_synthetic(20000)
res_syn_nl = comparison_with_baselines_nonlinear("Synthetic", (X_syn, y_syn), recourse_budget=BUDGET, folds=FOLDS, max_instances=INSTANCES, seed=SEEDS, T=T_STEPS)
all_results_nonlinear.append(res_syn_nl)

# 2. Polish
try:
    print("Processing Polish...")
    X_pol, y_pol = load('polish')
    res_pol_nl = comparison_with_baselines_nonlinear("Polish", (X_pol, y_pol), recourse_budget=BUDGET, folds=FOLDS, max_instances=INSTANCES, seed=SEEDS, T=T_STEPS)
    all_results_nonlinear.append(res_pol_nl)
except Exception as e:
    print(f"Skipping Polish: {e}")

# 3. COMPAS
try:
    print("Processing COMPAS...")
    X_com, y_com = load('compas')
    res_com_nl = comparison_with_baselines_nonlinear("COMPAS", (X_com, y_com), recourse_budget=BUDGET, folds=FOLDS, max_instances=INSTANCES, seed=SEEDS, T=T_STEPS)
    all_results_nonlinear.append(res_com_nl)
except Exception as e:
    print(f"Skipping COMPAS: {e}")


# 4. German
try:
    print("Processing German...")
    X_com, y_com = dl.load_germanc()
    res_com_nl = comparison_with_baselines_nonlinear("German", (X_com, y_com), recourse_budget=BUDGET, folds=FOLDS, max_instances=INSTANCES, seed=SEEDS, T=T_STEPS)
    all_results_nonlinear.append(res_com_nl)
except Exception as e:
    print(f"Skipping German: {e}")

# --- Separate and Display Tables ---
if all_results_nonlinear:
    final_table_nl = pd.concat(all_results_nonlinear)

    # Split columns based on keyword
    validity_cols = [c for c in final_table_nl.columns if "Validity" in c]
    loss_cols = [c for c in final_table_nl.columns if "Loss" in c]

    print("\n=== Non-Linear Robust Validity (% Accepted by All Models) ===")
    display(final_table_nl[validity_cols])

    print("\n=== Non-Linear Average Robust Loss (Minimizing Worst-Case) ===")
    display(final_table_nl[loss_cols])
else:
    print("No results generated.")

## Experiment 3: Cost Validity Trade off ##


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import data_processing.data_loading as dl

# Configuration
budget_grid = [1.0, 2.0, 3.0, 4.0, 5.0]
T_STEPS = 200
SEEDS = 42
FOLDS = 2
INSTANCES = 10

# Load COMPAS data
X_com, y_com = load('compas')

results_dict = {'ADV': [], 'ElliCE': [], 'ROAR': [], 'ORPM': []}

print("Starting non-linear sensitivity analysis on the COMPAS dataset...")
for b in budget_grid:
    print(f"\n--- Evaluating Budget: {b} ---")
    res_df = comparison_with_baselines_nonlinear("COMPAS", (X_com, y_com), recourse_budget=b, folds=FOLDS, max_instances=INSTANCES, seed=SEEDS, T=T_STEPS)

    for method in results_dict.keys():
        col_name = f"{method} Validity"
        if col_name in res_df.columns:
            val_str = res_df.iloc[0][col_name]
            if val_str != "N/A":
                val = float(val_str.split('%')[0])
                results_dict[method].append(val)
            else:
                results_dict[method].append(np.nan)
        else:
             results_dict[method].append(np.nan)

# Plotting
plt.figure(figsize=(10, 6))
markers = ['o', 's', '^', 'D']
linestyles = ['-', '--', '-.', ':']
for (method, values), marker, ls in zip(results_dict.items(), markers, linestyles):
    plt.plot(budget_grid, values, marker=marker, linestyle=ls, label=method, linewidth=2, markersize=8)

plt.xlabel('Recourse Budget (Delta)')
plt.ylabel('Robust Validity (%)')
plt.title('Non-Linear Cost-Validity Trade-off on COMPAS Dataset')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(budget_grid)
plt.ylim(-5, 105)
plt.show()


## Experiment 4: Scalability to Larger Competing Sets ##


In [ ]:
# Run the Scalability Experiment (Updated with Majority Vote Logic)
X_syn, y_syn = get_data_synthetic(20000)

model_configs = [
    (1, 3),   # 4 models
    (2, 6),   # 8 models
    (4, 12)   # 16 models
]

scalability_results = []
print("Starting Scalability Analysis on Synthetic Data (Majority Vote)...")

for n_lin, n_nonlin in model_configs:
    res_df = comparison_with_baselines_nonlinear_v2(
        "Synthetic", (X_syn, y_syn),
        num_linear=n_lin,
        num_nonlinear=n_nonlin,
        recourse_budget=3.0,
        folds=2,
        max_instances=10,  # Keep small for time reasons
        seed=42,
        T=200
    )
    scalability_results.append(res_df)

if scalability_results:
    df_scalability = pd.concat(scalability_results)
    display(df_scalability)
else:
    print("No results generated.")

In [ ]:
import pandas as pd
import numpy as np

def run_comprehensive_scalability_experiment(datasets_dict, model_configs, budgets, folds=1, max_instances=3, T=50):
    """
    Runs the scalability experiment across multiple datasets, model configurations, and budgets.

    Args:
        datasets_dict: dict of format {'DatasetName': (X, y)}
        model_configs: list of tuples (num_linear, num_nonlinear)
        budgets: list of float values for recourse budget

    Returns:
        A pandas DataFrame containing all results combined.
    """
    all_results = []

    for ds_name, ds_data in datasets_dict.items():
        print(f"\n=======================================================")
        print(f"Processing Dataset: {ds_name}")
        print(f"=======================================================")

        for (n_lin, n_nonlin) in model_configs:
            config_name = f"{n_lin + n_nonlin} Models ({n_lin}LR+{n_nonlin}NN)"
            print(f"\n  -> Config: {config_name}")

            for b in budgets:
                print(f"    -> Budget: {b}")
                # Call the existing baseline comparison function
                res_df = comparison_with_baselines_nonlinear_v2(
                    dataset_name=ds_name,
                    dataset=ds_data,
                    num_linear=n_lin,
                    num_nonlinear=n_nonlin,
                    recourse_budget=b,
                    folds=folds,
                    max_instances=max_instances,
                    seed=42,
                    T=T
                )

                # Add metadata columns for plotting
                res_df['Dataset'] = ds_name
                res_df['Config'] = config_name
                res_df['Total_Models'] = n_lin + n_nonlin
                res_df['Budget'] = b

                all_results.append(res_df)

    final_df = pd.concat(all_results, ignore_index=True)
    return final_df


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def clean_metric_column(df, col_name, is_percentage=False):
    """Helper to clean string columns to floats."""
    if is_percentage:
        return df[col_name].astype(str).str.replace('%', '', regex=False).replace('N/A', 'NaN').astype(float)
    return df[col_name].astype(str).replace('N/A', 'NaN').astype(float)

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def plot_scalability_vs_configs(results_df, metric='Validity', fixed_budget=2.0):
    """
    Plots the chosen metric as a function of the number of models (configs) with variance.
    """
    df_plot = results_df[results_df['Budget'] == fixed_budget].copy()
    if df_plot.empty:
        print(f"No data found for budget {fixed_budget}")
        return

    methods = ['ORPM', 'ADV', 'ElliCE', 'ROAR']
    datasets = df_plot['Dataset'].unique()
    fig, axes = plt.subplots(1, len(datasets), figsize=(6 * len(datasets), 5), squeeze=False)
    colors = plt.cm.tab10.colors

    for i, ds in enumerate(datasets):
        ax = axes[0, i]
        data = df_plot[df_plot['Dataset'] == ds].sort_values('Total_Models')
        x = data['Total_Models'].values

        for j, method in enumerate(methods):
            c = colors[j % len(colors)]
            if metric == 'Time':
                mean_col = f"{method} Time Mean (s)"
                std_col = f"{method} Time Std (s)"
            else:
                mean_col = f"{method} {metric} Mean"
                std_col = f"{method} {metric} Std"

            if mean_col in data.columns and std_col in data.columns:
                y_mean = pd.to_numeric(data[mean_col], errors='coerce').values
                y_std = pd.to_numeric(data[std_col], errors='coerce').values
                ax.plot(x, y_mean, label=method, marker='o', color=c, linewidth=2)
                ax.fill_between(x, y_mean - y_std, y_mean + y_std, color=c, alpha=0.2)

        ax.set_title(f"{ds} (Budget={fixed_budget})")
        ax.set_xlabel("Total Number of Models")
        ax.set_ylabel(metric)
        ax.grid(True, alpha=0.3)
        ax.set_xticks(x)
        if i == 0:
            ax.legend()

    plt.tight_layout()
    filename = f"scalability_vs_configs_{metric}_budget_{fixed_budget}.png"
    plt.savefig(filename, bbox_inches='tight')
    print(f"Saved plot to {filename}")
    plt.show()

def plot_scalability_vs_budget(results_df, metric='Validity'):
    """
    Plots the chosen metric as a function of budget, with subplots for each Dataset/Config combination, including variance.
    """
    df_plot = results_df.copy()
    methods = ['ORPM', 'ADV', 'ElliCE', 'ROAR']
    datasets = df_plot['Dataset'].unique()
    configs = df_plot['Config'].unique()
    colors = plt.cm.tab10.colors

    fig, axes = plt.subplots(len(configs), len(datasets), figsize=(6 * len(datasets), 5 * len(configs)), squeeze=False)

    for i, cfg in enumerate(configs):
        for j, ds in enumerate(datasets):
            ax = axes[i, j]
            data = df_plot[(df_plot['Dataset'] == ds) & (df_plot['Config'] == cfg)].sort_values('Budget')
            x = data['Budget'].values

            for k, method in enumerate(methods):
                c = colors[k % len(colors)]
                if metric == 'Time':
                    mean_col = f"{method} Time Mean (s)"
                    std_col = f"{method} Time Std (s)"
                else:
                    mean_col = f"{method} {metric} Mean"
                    std_col = f"{method} {metric} Std"

                if mean_col in data.columns and std_col in data.columns:
                    y_mean = pd.to_numeric(data[mean_col], errors='coerce').values
                    y_std = pd.to_numeric(data[std_col], errors='coerce').values
                    ax.plot(x, y_mean, label=method, marker='s', color=c, linewidth=2)
                    ax.fill_between(x, y_mean - y_std, y_mean + y_std, color=c, alpha=0.2)

            ax.set_title(f"{ds} | {cfg}")
            ax.set_xlabel("Recourse Budget")
            ax.set_ylabel(metric)
            ax.grid(True, alpha=0.3)
            ax.set_xticks(x)
            if i == 0 and j == 0:
                ax.legend()

    plt.tight_layout()
    filename = f"scalability_vs_budget_{metric}.png"
    plt.savefig(filename, bbox_inches='tight')
    print(f"Saved plot to {filename}")
    plt.show()

In [ ]:
import data_processing.data_loading as dl

experiment_datasets = {}


try:
    if 'load_data' in globals():
        X_syn, y_syn = load_data('synthetic')
    else:
        X_syn, y_syn = get_data_synthetic(2000)
    experiment_datasets['Synthetic'] = (X_syn, y_syn)
except Exception as e:
    print(f"Error loading Synthetic dataset: {e}")


try:
    if 'load_data' in globals():
        X_compas, y_compas = load_data('compas')
    else:
        X_compas, y_compas = dl.load_compas()
    experiment_datasets['COMPAS'] = (X_compas, y_compas)
except Exception as e:
    print(f"Error loading COMPAS dataset: {e}")

try:
    if 'load_data' in globals():
        X_german, y_german = load_data('german')
    else:
        X_german, y_german = dl.load_germanc()
    experiment_datasets['German'] = (X_german, y_german)
except Exception as e:
    print(f"Error loading German dataset: {e}")

try:
    if 'load_data' in globals():
        X_polish, y_polish = load_data('polish')
    else:
        X_polish, y_polish = dl.load_polish()
    experiment_datasets['Polish'] = (X_polish, y_polish)
except Exception as e:
    print(f"Error loading Polish dataset: {e}")


experiment_configs = [
    (1, 3), # 4 models
    (3, 9),  # 12 models
    (5, 15),  # 20 models
    (7, 21),  # 28 models
]

experiment_budgets = [1.0, 2.0, 3.0, 4.0, 5.0]

# Run the comprehensive experiment
print("Running comprehensive experiment...")
if not experiment_datasets:
    print("No datasets loaded. Please ensure dataset functions are defined.")
else:
    final_results_df = run_comprehensive_scalability_experiment(
        datasets_dict=experiment_datasets,
        model_configs=experiment_configs,
        budgets=experiment_budgets,
        folds=2,
        max_instances=20, # Very small for demonstration
        T=100
    )

    display(final_results_df)

    # Plot 1: Metric vs Configs (at a fixed budget of 2.0)
    print("\nPlotting Validity vs. Number of Models...")
    plot_scalability_vs_configs(final_results_df, metric='Validity', fixed_budget=2.0)

    # Plot 2: Metric vs Budget
    print("\nPlotting Validity vs. Budget...")
    plot_scalability_vs_budget(final_results_df, metric='Validity')


In [ ]:
plot_scalability_vs_configs(final_results_df, metric='Validity', fixed_budget=5.0)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def prepare_plot_data(df, metric='Validity'):
    """Melts and cleans the dataframe for seaborn plotting."""
    df_plot = df.copy()
    methods = ['ORPM', 'ADV', 'ElliCE', 'ROAR']
    value_vars = [f"{m} {metric}" for m in methods if f"{m} {metric}" in df_plot.columns]
    
    # Clean percentage strings to floats if needed
    for col in value_vars:
        if df_plot[col].dtype == object:
            df_plot[col] = df_plot[col].astype(str).str.replace('%', '', regex=False).replace('N/A', 'NaN').astype(float)
            
    df_melted = df_plot.melt(
        id_vars=['Dataset', 'Budget', 'Config', 'Total_Models'], 
        value_vars=value_vars, 
        var_name='Method', 
        value_name='Value'
    )
    # Clean method names (e.g., 'ORPM Validity' -> 'ORPM')
    df_melted['Method'] = df_melted['Method'].apply(lambda x: x.split(' ')[0])
    return df_melted

def plot_approach_1_validity_vs_budget(df, dataset_name=None, config_name=None):
    """Approach 1: Validity vs. Budget (Cost-Validity Trade-off)"""
    df_melted = prepare_plot_data(df, metric='Validity')

    df_filtered = df_melted
    if dataset_name:
        df_filtered = df_filtered[df_filtered['Dataset'] == dataset_name]
    if config_name:
        df_filtered = df_filtered[df_filtered['Config'] == config_name]

    if df_filtered.empty:
        print(f"No data found for dataset {dataset_name} and config {config_name}")
        return

    g = sns.relplot(
        data=df_filtered,
        x='Budget',
        y='Value',
        hue='Method',
        col='Dataset' if not dataset_name else None,
        row='Config' if not config_name else None,
        kind='line',
        marker='o',
        height=4,
        aspect=1.2
    )
    g.set_axis_labels("Recourse Budget (Delta)", "Robust Validity (%)")
    
    title_parts = []
    if dataset_name:
        title_parts.append(dataset_name)
    if config_name:
        title_parts.append(config_name)
    title_suffix = f" - {' | '.join(title_parts)}" if title_parts else ""
    
    g.fig.suptitle(f"Approach 1: Cost-Validity Trade-off{title_suffix}", y=1.05)

    filename_parts = ["approach1_cost_validity"]
    if dataset_name:
        filename_parts.append(dataset_name)
    else:
        filename_parts.append("all_datasets")
    if config_name:
        safe_config = config_name.replace(" ", "_").replace("(", "").replace(")", "").replace("+", "_")
        filename_parts.append(safe_config)
        
    filename = "_".join(filename_parts) + ".png"
    plt.savefig(filename, bbox_inches='tight')
    print(f"Saved plot to {filename}")

    plt.show()

def plot_approach_2_validity_vs_models(df, dataset_name=None, fixed_budget=2.0):
    """Approach 2: Validity vs. Number of Models (Robustness Scalability)"""
    df_melted = prepare_plot_data(df, metric='Validity')
    df_filtered = df_melted[df_melted['Budget'] == fixed_budget]
    
    if dataset_name:
        df_filtered = df_filtered[df_filtered['Dataset'] == dataset_name]
    
    if df_filtered.empty:
        print(f"No data found for budget {fixed_budget} and dataset {dataset_name}")
        return
        
    g = sns.relplot(
        data=df_filtered, 
        x='Total_Models', 
        y='Value', 
        hue='Method', 
        col='Dataset' if not dataset_name else None, 
        kind='line', 
        marker='s',
        height=4, 
        aspect=1.2
    )
    g.set_axis_labels("Total Number of Models", "Robust Validity (%)")
    title_suffix = f" - {dataset_name}" if dataset_name else ""
    g.fig.suptitle(f"Approach 2: Robustness Scalability (Budget={fixed_budget}){title_suffix}", y=1.05)
    
    # Ensure x-axis has integer ticks for models
    for ax in g.axes.flat:
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    
    filename = f"robustness_scalability_{dataset_name if dataset_name else 'all_datasets'}_budget_{fixed_budget}.png"
    plt.savefig(filename, bbox_inches='tight')
    print(f"Saved plot to {filename}")

    plt.show()

# Render the plots using the dataframe from the previous cell
if 'final_results_df' in globals() and not final_results_df.empty:
    plot_approach_1_validity_vs_budget(final_results_df, dataset_name='Synthetic', config_name='8 Models (2LR+6NN)')
    # Example: Plot only for the 'Synthetic' dataset
    plot_approach_2_validity_vs_models(final_results_df, dataset_name='Synthetic', fixed_budget=2.0)
else:
    print("final_results_df not found. Please run the scalability experiment first.")

In [ ]:
plot_approach_1_validity_vs_budget(final_results_df, dataset_name='COMPAS', config_name='4 Models (1LR+3NN)')

In [ ]:
plot_approach_2_validity_vs_models(final_results_df, dataset_name='COMPAS', fixed_budget=4.0)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def prepare_time_data(df):
    """Melts and cleans the dataframe specifically for Execution Time."""
    df_plot = df.copy()
    methods = ['ORPM', 'ADV', 'ElliCE', 'ROAR']
    time_cols = [f"{m} Time (s)" for m in methods if f"{m} Time (s)" in df_plot.columns]

    # Clean string values to floats if needed
    for col in time_cols:
        if df_plot[col].dtype == object:
            df_plot[col] = df_plot[col].astype(str).replace('N/A', 'NaN').astype(float)

    df_melted = df_plot.melt(
        id_vars=['Dataset', 'Budget', 'Config', 'Total_Models'],
        value_vars=time_cols,
        var_name='Method',
        value_name='Time (s)'
    )
    # Clean method names
    df_melted['Method'] = df_melted['Method'].apply(lambda x: x.split(' ')[0])
    return df_melted

def plot_approach_3_time_scalability(df, dataset_name=None, fixed_budget=2.0):
    """Approach 3: Computational Time vs. Number of Models"""
    df_melted = prepare_time_data(df)
    df_filtered = df_melted[df_melted['Budget'] == fixed_budget]

    if dataset_name:
        df_filtered = df_filtered[df_filtered['Dataset'] == dataset_name]

    if df_filtered.empty:
        print(f"No time data found for budget {fixed_budget}")
        return

    g = sns.relplot(
        data=df_filtered,
        x='Total_Models',
        y='Time (s)',
        hue='Method',
        col='Dataset' if not dataset_name else None,
        kind='line',
        marker='D',
        height=4,
        aspect=1.2
    )
    g.set_axis_labels("Total Number of Models", "Execution Time (s)")
    title_suffix = f" - {dataset_name}" if dataset_name else ""
    g.fig.suptitle(f"Computational Scalability (Budget={fixed_budget}){title_suffix}", y=1.05)

    # Use log scale for time, and integers for model counts
    for ax in g.axes.flat:
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.set_yscale('log')

    filename = f"time_scalability_{dataset_name if dataset_name else 'all_datasets'}.png"
    plt.savefig(filename, bbox_inches='tight')
    print(f"Saved plot to {filename}")
    plt.show()

def plot_approach_4_validity_heatmap(df, fixed_budget=2.0):
    """Approach 4: Heatmap Comparison across Datasets and Configs"""
    # Note: prepare_plot_data is defined in the previous cell (441a0364)
    df_melted = prepare_plot_data(df, metric='Validity') 
    df_filtered = df_melted[df_melted['Budget'] == fixed_budget].copy()

    if df_filtered.empty:
        print(f"No data found for budget {fixed_budget}")
        return

    # Create a combined identifier for the y-axis
    df_filtered['Scenario'] = df_filtered['Dataset'] + " | " + df_filtered['Config']

    pivot_df = df_filtered.pivot_table(index='Scenario', columns='Method', values='Value', aggfunc='mean')

    plt.figure(figsize=(8, len(pivot_df) * 0.8 + 2))
    sns.heatmap(pivot_df, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={'label': 'Robust Validity (%)'}, vmin=0, vmax=100)
    plt.title(f"Validity Comparison Heatmap (Budget={fixed_budget})")
    plt.ylabel("Dataset and Model Configuration")
    plt.xlabel("Method")
    plt.tight_layout()

    filename = f"validity_heatmap_budget_{fixed_budget}.png"
    plt.savefig(filename, bbox_inches='tight')
    print(f"Saved plot to {filename}")
    plt.show()

# Render the plots using the dataframe from the scalability experiment
if 'final_results_df' in globals() and not final_results_df.empty:
    # Example: Plot Time Scalability for Synthetic data, and the general Heatmap
    plot_approach_3_time_scalability(final_results_df, dataset_name='Synthetic', fixed_budget=2.0)
    plot_approach_4_validity_heatmap(final_results_df, fixed_budget=2.0)
else:
    print("final_results_df not found. Please run the scalability experiment first.")

## Experiment 5: Feature Scalability (High-Dimensional Data) ##


In [ ]:
from sklearn.datasets import make_classification
import pandas as pd
import matplotlib.pyplot as plt

# 1. Generate a high-dimensional dataset
print("Generating high-dimensional synthetic dataset (50 features)...")
X_hd, y_hd = make_classification(
    n_samples=20000,
    n_features=100,
    n_informative=80,
    n_redundant=0,
    random_state=42
)

high_dim_dataset = {'HighDim_50F': (X_hd, y_hd)}

# 2. Define budgets and configurations to test
# Limiting budget to two values and adding a larger configuration
hd_budgets = [1.0, 3.0, 5.0]
hd_configs = [(1, 3), (3, 9), (5,15)] # 4 models and 8 models

# 3. Run the experiment
print("\nRunning feature scalability experiment on high-dimensional data...")
hd_results_df = run_comprehensive_scalability_experiment(
    datasets_dict=high_dim_dataset,
    model_configs=hd_configs,
    budgets=hd_budgets,
    folds=2,
    max_instances=30, # Number of instances to generate recourse for
    T=200 # Number of iterations for internal optimization
)

display(hd_results_df)

# 4. Plot Validity vs Budget
if not hd_results_df.empty:
    print("\nPlotting results...")
    plot_approach_1_validity_vs_budget(hd_results_df, dataset_name='HighDim_50F')


In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
import random
import data_processing.data_loading as dl

# --- 2. Candidate Selection (Using Rashomon Set) ---
def train_and_select_candidates_rashomon(X, y, max_instances=10, seed=42, num_models=4):
    """
    Trains a central Logistic Regression model using 5-Fold CV, generates a Rashomon set,
    and selects candidate instances. Ensures models are Linear for ElliCE compatibility.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    central_accs = []
    rashomon_avg_accs = []
    final_models = []
    final_candidates = []

    print(f"Starting 5-Fold Cross-Validation (Rashomon Linear) with seed {seed}...")

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        # print(f"  CV Fold {fold+1}/5...")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        input_dim = X.shape[1]
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

        # 1. Train Central Model (Linear)
        central_model = LogisticRegressionModel(input_dim, seed=seed)
        train_model(central_model, X_train, y_train, epochs=500, lr=0.01, seed=seed)

        central_model.eval()
        with torch.no_grad():
            preds = (central_model(X_test_tensor) >= 0.5).float()
            acc = (preds == y_test_tensor).float().mean().item()
        central_accs.append(acc)

        # 2. Generate Rashomon Set
        # Augment X for Hessian (w, b)
        X_train_aug = np.hstack([X_train, np.ones((X_train.shape[0], 1))])

        w = central_model.linear.weight.detach().view(-1)
        b = central_model.linear.bias.detach().view(-1)
        central_theta = torch.cat([w, b])

        probs = central_model(X_train_tensor).squeeze().detach()
        S = probs * (1 - probs)
        X_aug_tensor = torch.tensor(X_train_aug, dtype=torch.float32)
        H = (X_aug_tensor.T * S) @ X_aug_tensor / X_train.shape[0]
        H = H + 1e-5 * torch.eye(H.shape[0])

        # Sample Models
        epsilon = 0.05
        thetas = sample_from_ellipsoid(central_theta, H, epsilon, num_models, seed=seed+fold)

        current_fold_models = []
        fold_rashomon_accs = []

        for i, theta in enumerate(thetas):
            m = LogisticRegressionModel(input_dim, seed=seed+i)
            set_model_weights(m, theta)
            m.eval()
            with torch.no_grad():
                preds = (m(X_test_tensor) >= 0.5).float()
                acc = (preds == y_test_tensor).float().mean().item()
            fold_rashomon_accs.append(acc)
            current_fold_models.append(m)

        rashomon_avg_accs.append(np.mean(fold_rashomon_accs))

        # 3. Last Fold Candidates
        if fold == 4:
            final_models = current_fold_models
            print("    Identifying candidates in Fold 5 where ALL models predict prob < 0.5...")
            with torch.no_grad():
                all_preds = []
                for model in final_models:
                    preds = model(X_test_tensor).squeeze()
                    all_preds.append(preds)
                all_preds = torch.stack(all_preds)
                is_negative = (all_preds < 0.5).all(dim=0)
                indices = is_negative.nonzero(as_tuple=True)[0]
                for idx in indices:
                    final_candidates.append(X_test_tensor[idx])
            print(f"    Found {len(final_candidates)} candidate instances in the last fold.")

    print(f"  CV Results: Central Acc={np.mean(central_accs):.3f}, Ensemble Acc={np.mean(rashomon_avg_accs):.3f}")

    if len(final_candidates) > max_instances:
        selected_instances = random.sample(final_candidates, max_instances)
    else:
        selected_instances = final_candidates

    return final_models, selected_instances


def run_generalization_experiment(dataset_name, dataset, num_opt_models=20, num_eval_models=20, recourse_budget=6.0, max_instances=30, seed=42, T=100, folds=2):
    print(f"\n=== Generalization Experiment: {dataset_name} ({folds}-Fold CV) ===")
    X, y = dataset
    scaler = StandardScaler()
    X_norm = scaler.fit_transform(X)

    total_models = num_opt_models + num_eval_models

    fold_results = {'ORPM_opt': [], 'ORPM_eval': [], 'ElliCE_opt': [], 'ElliCE_eval': []}

    for fold in range(folds):
        print(f"\n  --- Fold {fold + 1}/{folds} ---")
        current_seed = seed + fold

        all_models, candidates = train_and_select_candidates_rashomon(
            X_norm, y, 
            max_instances=max_instances, 
            seed=current_seed, 
            num_models=total_models
        )

        if len(all_models) <= num_opt_models:
            print("Not enough models generated for the evaluation set. Skipping fold.")
            continue

        # Split into optimization set and unseen evaluation set
        opt_models = all_models[:num_opt_models]
        eval_models = all_models[num_opt_models:]

        print(f"Using {len(opt_models)} models for optimization and {len(eval_models)} unseen models for evaluation.")

        results_validity = {'ORPM_opt': 0, 'ORPM_eval': 0, 'ElliCE_opt': 0, 'ElliCE_eval': 0}
        total = len(candidates)
        
        if total == 0:
            print("No valid candidates found in this fold.")
            continue

        for x0 in candidates:
            # Generate Recourse using ORPM on optimization models
            x_orpm, _, _, _, _ = solve_robust_recourse_with_oracle(
                x0, opt_models, recourse_budget, recourse_loss, oracle_gradient_descent, T=T, B=1
            )
            # Generate Recourse using ElliCE on optimization models
            hist_ellice = ElliCE(x0, opt_models, B=recourse_budget, loss_func=recourse_loss, T=T, lr=0.01)
            x_ellice = hist_ellice[-1]
            
            # Helper to check validity
            def check_validity(x_rec, models):
                with torch.no_grad():
                    for m in models:
                        if m(x_rec).view(-1).item() < 0.5:
                            return False
                return True
            
            # Check validity on optimization set (seen) vs evaluation set (unseen)
            if check_validity(x_orpm, opt_models): results_validity['ORPM_opt'] += 1
            if check_validity(x_orpm, eval_models): results_validity['ORPM_eval'] += 1
            if check_validity(x_ellice, opt_models): results_validity['ElliCE_opt'] += 1
            if check_validity(x_ellice, eval_models): results_validity['ElliCE_eval'] += 1

        fold_results['ORPM_opt'].append((results_validity['ORPM_opt'] / total) * 100)
        fold_results['ORPM_eval'].append((results_validity['ORPM_eval'] / total) * 100)
        fold_results['ElliCE_opt'].append((results_validity['ElliCE_opt'] / total) * 100)
        fold_results['ElliCE_eval'].append((results_validity['ElliCE_eval'] / total) * 100)

    if not fold_results['ORPM_opt']:
        return pd.DataFrame()

    summary = {
        'ORPM Opt Validity (%)': f"{np.mean(fold_results['ORPM_opt']):.2f} (+/- {np.std(fold_results['ORPM_opt']):.2f})",
        'ORPM Generalization (%)': f"{np.mean(fold_results['ORPM_eval']):.2f} (+/- {np.std(fold_results['ORPM_eval']):.2f})",
        'ElliCE Opt Validity (%)': f"{np.mean(fold_results['ElliCE_opt']):.2f} (+/- {np.std(fold_results['ElliCE_opt']):.2f})",
        'ElliCE Generalization (%)': f"{np.mean(fold_results['ElliCE_eval']):.2f} (+/- {np.std(fold_results['ElliCE_eval']):.2f})"
    }
    return pd.DataFrame(summary, index=[dataset_name])

# Run the experiment on Synthetic, COMPAS, Polish, and German datasets
gen_results = []

try:
    if 'X_syn' not in globals():
        X_syn, y_syn = get_data_synthetic(2000)
    res_syn = run_generalization_experiment("Synthetic", (X_syn, y_syn), recourse_budget=5.0, max_instances=10, folds=2)
    gen_results.append(res_syn)
except Exception as e:
    print(f"Skipping Synthetic dataset: {e}")

try:
    if 'X_com' not in globals():
        X_com, y_com = dl.load_compas_notebook()
    res_com = run_generalization_experiment("COMPAS", (X_com, y_com), recourse_budget=5.0, max_instances=10, folds=2)
    gen_results.append(res_com)
except Exception as e:
    print(f"Skipping COMPAS dataset: {e}")

try:
    if 'X_pol' not in globals():
        X_pol, y_pol = dl.load_polish_notebook()
    res_pol = run_generalization_experiment("Polish", (X_pol, y_pol), recourse_budget=5.0, max_instances=10, folds=2)
    gen_results.append(res_pol)
except Exception as e:
    print(f"Skipping Polish dataset: {e}")

try:
    if 'X_ger' not in globals():
        X_ger, y_ger = dl.load_germanc_notebook()
    res_ger = run_generalization_experiment("German", (X_ger, y_ger), recourse_budget=5.0, max_instances=10, folds=2)
    gen_results.append(res_ger)
except Exception as e:
    print(f"Skipping German dataset: {e}")

if gen_results:
    final_gen_df = pd.concat(gen_results)
    display(final_gen_df)
else:
    print("No results to display.")